# Solution 2.6: Loading, Inspecting and Cleaning (Angola IEA)

`IEA_2025_IV_TRIM_IND.sav`: the Inquerito ao Emprego em Angola, 4th quarter 2025,
published by INE Angola. 53,353 people, 206 columns, labels in Portuguese.

You will practice: reading the codebook to choose columns, renaming them to
something readable, loading with value labels, recasting what the labels got
wrong, and measuring what is missing.

**PT:** `IEA_2025_IV_TRIM_IND.sav`: Inquerito ao Emprego em Angola, IV trimestre
2025, publicado pelo INE Angola. 53.353 pessoas, 206 colunas, etiquetas em
portugues.

Vai praticar: ler o dicionario de variaveis para escolher colunas, renomea-las
para algo legivel, carregar com etiquetas de valores, corrigir os tipos que as
etiquetas estragaram, e medir o que esta em falta.

> **Pipeline:** reads `0_raw/`, writes `10_cleaned/`.

### Path Setup (run first)

Define the country folder once, then join sub folder and file name onto it.

**PT:** Defina a pasta do pais uma vez e depois junte a subpasta e o nome do
ficheiro.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Country folder first, then join onwards
# Primeiro a pasta do pais, depois juntar o resto
DATA_RAW_DIR = '../../data/0_raw'
DATA_CLEAN_DIR = '../../data/10_cleaned'

EMPLOYMENT_DIR = 'employment_survey'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, EMPLOYMENT_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Read the codebook

206 columns named `ATW_PAY` or `SRH_AVN` tell you nothing. The file carries a
description for every one, so read that first.

`pd.read_spss` does not expose these descriptions, so this cell uses `pyreadstat`
with `metadataonly=True`, which reads the header without loading any data.

> Just run this cell and the next one. They set up the codebook you will use to
> choose columns in Task 3.

**PT:** 206 colunas com nomes como `ATW_PAY` nao dizem nada. O ficheiro traz uma
descricao de cada uma. `pd.read_spss` nao da acesso a estas descricoes, por isso
esta celula usa `pyreadstat` com `metadataonly=True`, que le o cabecalho sem
carregar dados.

> Basta correr esta celula e a seguinte. Preparam o dicionario que vai usar para
> escolher colunas na Tarefa 3.

In [ ]:
import pyreadstat

# Header only, no data loaded / Apenas o cabecalho, sem carregar dados
_, meta = pyreadstat.read_sav(raw_path, metadataonly=True)
codebook = meta.column_names_to_labels

print('Variables described in the file:', len(codebook))
for name in ['PROV', 'DEM_AGE', 'ATW_PAY', 'SRH_AVN']:
    print(f'{name:12s} {codebook[name][:60]}')

---

## Task 2: Find columns by what they measure

Search the descriptions instead of guessing at names. The number of matches is
itself informative: a precise concept returns one column, a vague one returns
twenty and forces you to read.

**PT:** Pesquise nas descricoes em vez de adivinhar nomes. O numero de
resultados ja e informativo: um conceito preciso devolve uma coluna, um conceito
vago devolve vinte e obriga a ler.

**What to do:** complete `find_columns` so it returns the name and description
of every column whose description contains a keyword, then run it for a few
keywords and read the matches.

**O que fazer:** complete `find_columns` para devolver o nome e a descricao de
cada coluna cuja descricao contem uma palavra, e depois leia os resultados.

In [ ]:
def find_columns(codebook, keyword):
    """Columns whose description contains `keyword`, case insensitive.

    Colunas cuja descricao contem `keyword`, ignorando maiusculas.
    """
    return {name: label for name, label in codebook.items()
            if label and keyword.lower() in label.lower()}


for keyword in ['sexo', 'idade', 'provincia', 'horas']:
    print(f'{keyword:12s} {len(find_columns(codebook, keyword)):3d} matches')

In [ ]:
# A vague keyword needs reading, not trusting
# Uma palavra vaga precisa de ser lida, nao de confianca cega
for name, label in find_columns(codebook, 'idade').items():
    print(f'{name:16s} {label[:70]}')

**Answers:**

- `sexo` and `provincia` match one column each. `idade` matches 22 and `horas`
  29, because those words appear inside long question texts about other things.
- Of the 22 `idade` matches only `DEM_AGE` is the respondent's own age. `G_13`
  counts household members aged 15 or more, a different variable entirely.
- Searching the codebook narrows 206 columns to a handful you then read properly.

**PT:** `sexo` e `provincia` devolvem uma coluna cada. `idade` devolve 22 e
`horas` 29, porque essas palavras aparecem dentro de perguntas sobre outras
coisas. Das 22, so `DEM_AGE` e a idade do proprio inquirido. Pesquisar o
dicionario reduz 206 colunas a um punhado que depois se le com atencao.

---

## Task 3: Select the columns and give them readable names

The questionnaire's names are precise and unreadable. `SRH_AVN` is correct and
tells you nothing; `available_last_week` tells you what it holds.

Rename once, here, and every later line of code is easier to check. The mapping
is the bridge back to the official codebook, so it gets saved in Task 4 rather
than living only in this notebook.

**PT:** Os nomes do questionario sao precisos e ilegiveis. `SRH_AVN` esta certo e
nao diz nada; `available_last_week` diz o que contem.

Renomeie uma vez, aqui, e todas as linhas seguintes ficam mais faceis de
verificar. O mapeamento e a ponte de volta ao dicionario oficial, por isso e
gravado na Tarefa 4.

**What to do:** read `RENAME_MAP` below, then take its keys as the list of
columns to load.

**O que fazer:** leia o `RENAME_MAP` abaixo e use as suas chaves como lista de
colunas a carregar.

In [ ]:
# Questionnaire name -> readable name / Nome do questionario -> nome legivel
RENAME_MAP = {
    'NIDF': 'household_id',
    'PPNO': 'person_id',
    'G_06_ID_IEA': 'cluster_id',
    'PROV': 'province',
    'AREA_RESID': 'area_type',
    'G_15_TRIMESTRE': 'quarter',
    'DEM_REL': 'relation_to_head',
    'DEM_SEX': 'sex',
    'DEM_AGE': 'age',
    'DEM_MRT': 'marital_status',
    'DEM_EDL': 'education_level',
    'S03_01': 'attended_school',
    'ATW_PAY': 'worked_for_pay',
    'ATW_PFT': 'worked_own_account',
    'ATW_FAM': 'worked_family_business',
    'ABS_JOB': 'absent_from_job',
    'SRH_JOB': 'sought_job',
    'SRH_BUS': 'sought_business',
    'SRH_AVN': 'available_last_week',
    'SRH_AVL': 'available_next_2weeks',
    'SRH_DES': 'wants_work',
    'WKT_USHRSTOT': 'usual_hours',
    'WKT_ACHRSTOT': 'actual_hours',
    'MJT_SYR': 'job_start_year',
    'MJJ_EMP_REL': 'employment_relation',
    'GHVEDT': 'interview_date',
    'POND_IEA_IV_TRIM_2025_IND': 'weight',
}

SPSS_COLS = list(RENAME_MAP)
print('Selected:', len(SPSS_COLS), 'of', len(codebook))

**Answers:**

- 27 of 206. `G_12` and `G_13` are deliberately absent: they are empty in every
  row of this extract, so there is no reason to load them.
- One dictionary does two jobs: its keys are the columns to load, its values the
  names to use afterwards, so the two can never drift apart.
- Renaming is a convenience with a cost: `available_last_week` no longer matches
  anything in INE's documentation. Task 4 saves the mapping so the link survives.

**PT:** 27 de 206. `G_12` e `G_13` ficam de fora porque estao vazias em todas as
linhas. Um dicionario faz duas coisas: as chaves sao as colunas a carregar, os
valores os nomes a usar. Renomear tem um custo: `available_last_week` ja nao
corresponde a nada na documentacao do INE, por isso a Tarefa 4 grava o
mapeamento.

---

## Task 4: Load, rename, and save the codebook

`pd.read_spss` applies the file's value labels by default, so coded variables
arrive as readable Portuguese text.

Then save the three way mapping, original name, new name and description, as its
own small file. Six months from now it is the only thing that will tell you what
`wants_work` was called and what question produced it.

**PT:** `pd.read_spss` aplica as etiquetas de valores por omissao, por isso as
variaveis codificadas chegam como texto legivel em portugues.

Depois grave o mapeamento de tres colunas, nome original, nome novo e descricao,
num ficheiro proprio. Daqui a seis meses e a unica coisa que lhe dira como se
chamava `wants_work` e que pergunta a produziu.

**What to do:** load the selected columns, rename them with `RENAME_MAP`, then
build a frame with the original name, the new name and the description, and
save it. Remember the dtypes as loaded, because Task 8 needs them.

**O que fazer:** carregue as colunas escolhidas, renomeie com `RENAME_MAP`,
construa uma tabela com nome original, nome novo e descricao, e grave-a.

In [ ]:
df = pd.read_spss(raw_path, usecols=SPSS_COLS)
df = df.rename(columns=RENAME_MAP)

print('Loaded:', df.shape)
df.head()

In [ ]:
# Keep the bridge back to the official documentation
# Guardar a ponte de volta a documentacao oficial
codebook_df = pd.DataFrame({
    'original_name': SPSS_COLS,
    'new_name': [RENAME_MAP[name] for name in SPSS_COLS],
    'description': [codebook[name] for name in SPSS_COLS],
})

# Remember how each column arrived, so Task 8 can record what changed
# Guardar como cada coluna chegou, para a Tarefa 8 registar o que mudou
dtypes_on_load = df.dtypes.astype('string')

os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df.to_csv(codebook_path, index=False)

print('Saved:', codebook_path)
codebook_df.head(8)

**Answers:**

- 53,353 rows and 27 columns, one row per person.
- `province`, `area_type` and `sex` arrive as readable text (`Luanda`, `Urbana`,
  `Feminino`) rather than the numeric codes stored in the file.
- The codebook file is small and boring and will save somebody hours. It is the
  only artefact that survives the rename, because the CSV of data carries names
  with no explanation attached.

**PT:** 53.353 linhas e 27 colunas, uma por pessoa. `province`, `area_type` e
`sex` chegam como texto legivel em vez dos codigos numericos. O ficheiro de
dicionario e pequeno e aborrecido e vai poupar horas a alguem: e o unico artefacto
que sobrevive a mudanca de nomes.

---

## Task 5: Summary statistics

Look hard at every `min` and `max`, and at any `count` below 53,353.

**PT:** Olhe com atencao para cada `min` e `max`, e para qualquer `count` abaixo
de 53.353.

**What to do:** run `describe(include='all')` and read the result.

**O que fazer:** corra `describe(include='all')` e leia o resultado.

In [ ]:
df.describe(include='all').T

**Answers:**

- `usual_hours` and `actual_hours` max out at 997. Nobody works 997 hours in a
  week, so that is a code standing in for an answer, not a measurement.
- `age` runs 0 to 120. Age 0 is legitimate, 1,532 infants. Age 120 is not.
- `interview_date` has a mean around 20,251,000, nonsense as a number because it
  is really a date written as the digits `20251204`.
- The `count` row exposes the skip pattern: the labour columns are answered by
  about a fifth of the sample.

**PT:** `usual_hours` e `actual_hours` chegam a 997: e um codigo, nao uma medida.
`age` vai de 0 a 120; 0 e legitimo, 120 nao. `interview_date` e uma data escrita
como digitos. A linha `count` revela os saltos do questionario.

---

## Task 6: Explore categories

`value_counts()` shows what is actually in a coded column. Always pass
`dropna=False`. A bar plot makes the shape obvious at a glance.

**PT:** `value_counts()` mostra o que esta realmente numa coluna codificada. Use
sempre `dropna=False`. Um grafico de barras torna a forma imediata.

**What to do:** count the provinces, plot them as a horizontal bar chart sorted
by size, then count the other categorical columns and the households.

**O que fazer:** conte as provincias, desenhe um grafico de barras horizontais
ordenado, e depois conte as outras colunas categoricas e os agregados.

In [ ]:
print(df['province'].value_counts(dropna=False).sort_index())

In [ ]:
# Sort before plotting so the bars carry the ranking
# Ordenar antes de desenhar para que as barras mostrem o ranking
df['province'].value_counts().sort_values().plot(
    kind='barh', figsize=(8, 7), color='steelblue')
plt.title('People interviewed by province')
plt.xlabel('People / Pessoas')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
print(df['area_type'].value_counts(dropna=False))
print()
print(df['sex'].value_counts(dropna=False))
print()
print('Distinct households:', df['household_id'].nunique())
print(df['household_id'].value_counts().describe())

**Answers:**

- All 21 provinces appear, Luanda largest at 4,424. Icolo e Bengo, Moxico Leste
  and Cuando were created in 2024, so any older reference table will not know
  them.
- The bar plot shows the sample is spread deliberately rather than by population:
  Luanda is the largest but nothing like 4,424 against 1,498 would suggest if
  these were population shares. That is the survey design, and it is why the
  weight column exists.
- 34,173 urban and 19,180 rural; 25,601 male and 27,752 female.
- 13,036 households across 53,353 people, a mean of 4.09 each.

**PT:** Aparecem as 21 provincias, Luanda a maior com 4.424. Icolo e Bengo,
Moxico Leste e Cuando foram criadas em 2024. O grafico mostra que a amostra e
distribuida por desenho e nao por populacao, e por isso existe a coluna de
ponderador. 13.036 agregados para 53.353 pessoas, media de 4,09.

---

## Task 7: Read the dtypes critically

The dtypes decide the next hour of work.

**PT:** Os tipos de dados determinam a proxima hora de trabalho.

**What to do:** print the type of the frame and of one column, then print every
dtype and find the ones that do not match what the column means.

**O que fazer:** imprima o tipo da tabela e de uma coluna, depois todos os
tipos, e encontre os que nao correspondem ao significado da coluna.

In [ ]:
print(type(df))
print(type(df['age']))

In [ ]:
df.dtypes

**Answers:**

- Labelled columns are `category`, the rest `float64`.
- Four kinds are wrong for what the column means: `household_id`, `person_id`
  and `cluster_id` are identifiers arriving as floats with a trailing `.0`;
  `interview_date` is a date in a float; `job_start_year` is a **year** that came
  back as `category`; and `age` is a whole number of years sitting in a float.

**PT:** As colunas com etiquetas sao `category`, as restantes `float64`. Quatro
tipos estao errados: os identificadores chegam como float com `.0` no fim,
`interview_date` e uma data dentro de um float, `job_start_year` e um **ano**
que veio como `category`, e `age` e um numero inteiro de anos dentro de um float.

---

## Task 8: Recast what the file got wrong

`job_start_year` is the year somebody started their main job, and it arrived as a
category. Look at its categories to see why: alongside the years there is a text
label, so pandas concluded the whole column was categorical. A year you cannot
subtract is useless.

`pd.to_numeric` with `errors='coerce'` fixes it in one move: real years convert,
the text label becomes `NaN`. A year is a whole number, so store it as a nullable
integer, and summarise it with the median rather than the mean, since a year has
no meaningful average once part of the column is unknown.

**PT:** `job_start_year` e o ano em que a pessoa comecou o emprego principal, e
chegou como categoria. Veja as categorias: alem dos anos ha uma etiqueta de
texto, por isso o pandas tornou a coluna categorica. Um ano que nao se pode
subtrair nao serve.

`pd.to_numeric` com `errors='coerce'` resolve: os anos reais convertem, a
etiqueta vira `NaN`. Um ano e um numero inteiro, entao guarde como inteiro que
aceita nulos, e resuma com a mediana em vez da media.

**What to do:** look at what is in `job_start_year` besides years, cast it to a
nullable integer, cast `age` to a plain integer, replace the hour codes, turn the
identifiers into text and the interview date into a datetime. Finally add both
dtype columns to the codebook and save it again.

**O que fazer:** veja o que ha em `job_start_year` alem de anos, converta para
inteiro que aceita nulos, converta `age` para inteiro simples, substitua os
codigos das horas, passe os identificadores a texto e a data a datetime. No fim
junte as duas colunas de tipo ao dicionario e grave outra vez.

In [ ]:
# What is in there besides years? / O que ha alem de anos?
print(df['job_start_year'].value_counts().head(6))

In [ ]:
df['job_start_year'] = (
    pd.to_numeric(df['job_start_year'].astype('object'), errors='coerce')
    .astype('Int64')
)

print('dtype:  ', df['job_start_year'].dtype)
print('range:  ', df['job_start_year'].min(), 'to', df['job_start_year'].max())
print('median: ', df['job_start_year'].median())
print('unknown:', df['job_start_year'].isna().sum())

In [ ]:
# age is a whole number of years with no missing values, so plain int64 works.
# job_start_year above needed the nullable Int64 because it does have gaps.
# age e um numero inteiro de anos sem valores em falta, por isso int64 simples chega.
# job_start_year precisou de Int64 porque tem falhas.
print('age missing values:', df['age'].isna().sum())

df['age'] = df['age'].astype('int64')

print('age dtype:', df['age'].dtype, '| range:', df['age'].min(), 'to', df['age'].max())

In [ ]:
# The hours columns kept the same kind of code, but with no label attached,
# so they stayed numeric and it has to go by hand.
# As colunas de horas mantiveram o mesmo tipo de codigo, mas sem etiqueta,
# por isso continuam numericas e tem de ser tratadas a mao.
print('hours max before:', df['usual_hours'].max())

df['usual_hours'] = df['usual_hours'].replace([997, 998, 999], np.nan)
df['actual_hours'] = df['actual_hours'].replace([997, 998, 999], np.nan)

print('hours max after: ', df['usual_hours'].max())

In [ ]:
# Identifiers are labels, not quantities: float to integer to string, or the
# trailing .0 survives and joins to nothing.
# Identificadores sao etiquetas, nao quantidades: float para inteiro para texto.
print('Before:', df['household_id'].head(3).tolist())

for col in ['household_id', 'person_id', 'cluster_id']:
    df[col] = df[col].astype('int64').astype('string')

print('After: ', df['household_id'].head(3).tolist())

In [ ]:
# interview_date is the float 20251204.0. Int64 tolerates the missing values.
# interview_date e o float 20251204.0. Int64 aceita os valores em falta.
df['interview_date'] = pd.to_datetime(
    df['interview_date'].astype('Int64').astype('string'),
    format='%Y%m%d', errors='raise')

print('dtype:', df['interview_date'].dtype)
print('Range:', df['interview_date'].min(), 'to', df['interview_date'].max())
print()
print(df['interview_date'].dt.month.value_counts(dropna=False).sort_index())

In [ ]:
# Add both types to the codebook and save it again. A later notebook can then
# read this one file and restore every dtype without repeating the work above.
# Juntar os dois tipos ao dicionario e gravar de novo. Um caderno posterior pode
# ler este ficheiro e restaurar todos os tipos sem repetir o trabalho acima.
codebook_df['dtype_loaded'] = codebook_df['new_name'].map(dtypes_on_load)
codebook_df['dtype_final'] = codebook_df['new_name'].map(df.dtypes.astype('string'))

changed = codebook_df[codebook_df['dtype_loaded'] != codebook_df['dtype_final']]
print('Columns whose type changed / Colunas cujo tipo mudou:', len(changed))
print(changed[['new_name', 'dtype_loaded', 'dtype_final']].to_string(index=False))

codebook_df.to_csv(codebook_path, index=False)
print()
print('Codebook now carries the types:', list(codebook_df.columns))

**Answers:**

- `value_counts()` puts the odd one first: a text label meaning "does not know",
  with 1,673 rows. One non numeric value among the years is what forced pandas
  to treat the whole column as categorical.
- After the cast `job_start_year` is `Int64`, running 1965 to 2025 with a median
  of 2020 and 43,389 unknown. The median is the honest summary: a mean over years
  invites the reader to treat it as a date, and it is not one.
- `usual_hours` drops from 997 to 120 once the codes are replaced.
- Without the `int64` step the identifier reads `'9250068.0'` and matches nothing.
- Six columns changed type: the three identifiers became text, the date became a
  datetime, the year became a nullable integer, and age became a plain integer.
- `age` uses `int64` and `job_start_year` uses `Int64`, and the difference is not
  style: age has no missing values, while the year has 43,389. Plain `int64`
  cannot hold a gap, so a column with one needs the nullable form. The hours columns kept
  their type, because only their values were corrected. The codebook now
  records both types, so the file says which columns needed intervention.
- The date range is **2024-11-10 to 2026-01-28**, both ends impossible for a 4th
  quarter file. 9,026 interviews fall in November 2025 and 20,100 in December,
  553 in January 2026 and 3 in November 2024. Not one is dated October, which for
  a quarter running October to December is worth raising with the producer.

**PT:** A unica categoria que nao e um ano e uma etiqueta de texto que significa
"nao sabe". Depois da conversao, `job_start_year` e `Int64`, de 1965 a 2025, com
mediana de 2020 e 43.389 desconhecidos. A mediana e o resumo honesto. `age` usa `int64` e `job_start_year` usa `Int64`:
a diferenca e que a idade nao tem valores em falta e o ano tem 43.389. As horas
descem de 997 para 120. Sem o passo `int64` o identificador fica `'9250068.0'`. O
intervalo de datas vai de 2024-11-10 a 2026-01-28, e nenhuma entrevista tem data
de outubro, o que num IV trimestre merece ser reportado ao produtor.

---

## Task 9: Detect missing values

Count them, express them as a share, and look at the shape before deciding
anything.

**PT:** Conte, converta em percentagem, e observe o padrao antes de decidir.

**What to do:** build a frame with the count and the percentage of missing
values per column, sort it, then plot the columns that have any.

**O que fazer:** construa uma tabela com a contagem e a percentagem de valores
em falta por coluna, ordene, e desenhe as que tem falhas.

In [ ]:
missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
})
missing.sort_values('pct_missing', ascending=False)

In [ ]:
counts = df.isna().sum()
counts[counts > 0].sort_values().plot(kind='barh', color='coral', figsize=(9, 6))
plt.title('Missing values by column')
plt.xlabel('Count / Contagem')
plt.tight_layout()
plt.show()

**Answers:**

- A large block sits between 44% and 98.5% missing, and almost none of it is
  damage.
- The heavily missing group is the labour module: only people routed into it
  answer. `available_next_2weeks` is 98.5% missing because it is asked only of
  those who said no to `available_last_week`.
- Missing by design and missing by error need opposite treatment, so `dropna()`
  here would be a catastrophe: no single person answered every module.
- Check the codebook file for any column you are unsure about. The question text
  usually explains the gap.

**PT:** Um grande bloco esta entre 44% e 98,5% em falta, e quase nada disso e
dano. O grupo mais vazio e o modulo de trabalho: so responde quem o questionario
encaminha para la. `available_next_2weeks` esta 98,5% vazia porque so e
perguntada a quem disse nao a `available_last_week`. Falta por desenho e falta
por erro exigem tratamentos opostos.

---

## Task 10: Save

Raw data is read only. Write to `10_cleaned/` and reload to confirm the round
trip.

**PT:** Os dados brutos sao apenas de leitura. Grave em `10_cleaned/` e
recarregue para confirmar.

**What to do:** save the frame to `10_cleaned/` with `index=False`, then reload
it and look at which dtypes survived.

**O que fazer:** grave em `10_cleaned/` com `index=False`, recarregue, e veja
que tipos sobreviveram.

In [ ]:
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'household_id': 'string',
                                     'person_id': 'string',
                                     'cluster_id': 'string'})
print('Reloaded:', check.shape)
print(check[['household_id', 'province', 'interview_date', 'job_start_year']].dtypes)

**Answers:**

- 53,353 rows by 27 columns. Nothing was removed: this notebook fixed types and
  measured what is missing, it did not decide what to discard.
- `interview_date` reloads as text, `province` as text rather than `category`, and
  `job_start_year` as float rather than `Int64`. CSV stores no dtypes, so every
  reader re-establishes them.
- That is exactly why the codebook file written in Task 4 matters: it is the only
  part of this work that survives the round trip intact.

**PT:** 53.353 linhas por 27 colunas. Nada foi removido: este caderno corrigiu
tipos e mediu o que falta, nao decidiu o que descartar. Ao recarregar, os tipos
perdem-se, porque o CSV nao guarda tipos. E por isso que o ficheiro de dicionario
da Tarefa 4 importa: e a unica parte deste trabalho que sobrevive intacta.